# 14 · Where preferences come from — for free

**Recap (13):** preference tuning needs **(prompt, chosen, rejected)** triples.
Normally a *human* labels which answer is better — slow and expensive. This lab uses
a trick that needs **no humans and no separate reward model**: the database itself
judges.

In [ ]:
# This mirrors eval_harness.execution_reward in the repo.
def execution_reward(outcome):
    # run the predicted SQL, compare its RESULT to the gold result:
    return {"match": 1.0, "runs_but_wrong": 0.3, "errors": 0.0}[outcome]

print("reward tiers:", {o: execution_reward(o)
                        for o in ["match", "runs_but_wrong", "errors"]})

## Step 1 — The execution reward is an automatic judge

The lab already scores SQL by **running it** and comparing result sets (that's
`eval_harness.execution_reward`): `1.0` if it matches gold, `0.3` if it runs but is
wrong, `0.0` if it errors. No opinion, no human — just *does the SQL produce the
right answer?*

## Step 2 — Build a pair: sample candidates, score them, take best vs worst

To make a preference pair we sample several candidate SQLs from the model, score each
with the execution reward, then call the **highest-reward one `chosen`** and a
**low-reward one `rejected`**. That's exactly `data.make_preference_pairs`.

In [ ]:
# candidate SQLs sampled from the model, each run against the real store.db:
candidates = [
    ("SELECT COUNT(*) FROM customers", "match"),
    ("SELECT * FROM customers",        "runs_but_wrong"),
    ("SELECT COUNT(*) FROM custmers",  "errors"),
]

scored = sorted(((sql, execution_reward(o)) for sql, o in candidates),
                key=lambda t: t[1], reverse=True)
for sql, r in scored:
    print(f"reward {r:>3}  |  {sql}")

chosen, rejected = scored[0][0], scored[-1][0]
print("\nchosen   ->", chosen)
print("rejected ->", rejected)

## Step 3 — Why this is the elegant part

The **same** execution check that *evaluates* the model also *supplies its training
signal*. So: no human labels, no separate reward model, and the data is built and
verified **on CPU**, locally, before any GPU time. (Caveat: it works because SQL has a
*checkable* ground truth — you couldn't do this for "write a nicer poem.")

## Step 4 — A richer signal: *optimized* vs *resource-heavy* SQL

Here's a blind spot in the reward above. Two queries that return the **same rows** both
score `1.0` — it's blind to *how* they got there. So among all the **correct** answers,
DPO has **no preference signal at all**. But ranking equally-correct answers by a
secondary quality is *exactly* where preference tuning beats plain imitation.

We can measure SQL cost with SQLite's **`EXPLAIN`**, which counts the query plan's
bytecode operations — a deterministic proxy for "work" that registers even on a tiny
table (a needless cross-join compiles to more ops). Watch two **correct** queries get
different costs:

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")          # a throwaway DB, self-contained
con.executescript('''
    CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT);
    INSERT INTO customers VALUES (1,'Asha','Pune'),(2,'Ravi','Mumbai'),
                                 (3,'Priya','Pune'),(4,'Meera','Pune');
''')

def result(sql): return frozenset(con.execute(sql).fetchall())
def cost(sql):   return len(con.execute("EXPLAIN " + sql).fetchall())   # # of plan ops

optimized = "SELECT name FROM customers WHERE city='Pune'"
heavy     = ("SELECT c1.name FROM customers c1, customers c2 "
             "WHERE c1.city='Pune' AND c1.id=c2.id")      # needless self cross-join

print("same result set? ", result(optimized) == result(heavy))
print("optimized cost  :", cost(optimized), "ops")
print("heavy cost      :", cost(heavy), "ops")

Same answer, different cost. The rule for using this is **lexicographic — correctness
first, cost only as a tiebreak among correct queries**:

- `chosen` = most correct, and among the correct ones, the **cheapest**.
- `rejected` = the worst.
- **Never** prefer a cheap *wrong* query over an expensive *right* one — correctness
  always dominates.

So when both candidates are correct, cost decides:

In [ ]:
def reward(correct): return 1.0 if correct else 0.0

# both candidates correct -> the cheaper plan becomes `chosen`
cands = [
    (optimized, reward(True), cost(optimized)),
    (heavy,     reward(True), cost(heavy)),
]
cands.sort(key=lambda c: (c[1], -c[2]), reverse=True)   # reward desc, then cheaper first
print("chosen   (correct & cheapest):", cands[0][0])
print("rejected (correct & heaviest):", cands[-1][0])

This is now wired into the lab: **`eval_harness.sql_cost`** computes the proxy, and
**`data.make_preference_pairs`** uses the lexicographic rule above — it keeps a pair on
a *correctness gap* OR on a *cost gap between two correct queries*. Crucially,
**execution accuracy (the eval metric) is untouched** — it stays pure result-set
correctness, so the comparison across methods remains honest. Cost only shapes the
*training* signal.

**Honest caveat:** this only bites if the candidate set actually *contains* both cheap
and expensive correct queries — which means sampling candidates from the SFT model
(they'll vary in style), the intended next step in the data pipeline. And "optimized"
here means *SQLite's planner thinks it's cheaper*, not a universal truth.

## Recap

- **execution reward**: `1.0` match / `0.3` runs-but-wrong / `0.0` error — a free judge.
- A **preference pair** = best candidate (`chosen`) vs worst (`rejected`), no humans.
- **Among equally-correct candidates, cheaper SQL wins** (`sql_cost`, lexicographic) —
  the "optimized vs resource-heavy" signal that pure imitation can't express.
- The eval metric stays correctness-only; cost only enriches the *training* pairs.

**BAM!** Next: **`15 — modeling "A beats B"`.**